# Advanced Computer Vision Assignment 2
## Transfer learning, feature clustering, pose estimation, and deployment on Yoga-107

**Yan Zhou**  
This notebook reuses the exact cleaned 64/16/20 split from Assignment 1. Model selection uses validation data only; the held-out test set is evaluated after the configuration is fixed. `RUN_LONG_EXPERIMENTS=False` loads the recorded artifacts used in this submission. Set it to `True` to retrain from scratch.

In [ ]:
# Reproducible configuration
from pathlib import Path
import os, json, time, hashlib, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Image as DisplayImage
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, silhouette_score, adjusted_rand_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

SEED = 42
RUN_LONG_EXPERIMENTS = False
ROOT = Path.cwd()
if not (ROOT / 'artifacts').exists() and (ROOT / 'assignment2/artifacts').exists():
    ROOT = ROOT / 'assignment2'
ART = ROOT / 'artifacts'
DATA = ROOT / 'data/yoga107_raw/dataset'
keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| Root', ROOT)


## 1 Data and split

The source is the Kaggle Yoga Pose Image Classification Dataset. I decode and orient images, convert to RGB, hash dimensions plus decoded pixels, remove conflicting-label exact duplicates, and keep one same-label copy before splitting. The same random seed and stratified split from Assignment 1 give 3,712 training, 929 validation, and 1,161 test images across 107 classes. Near duplicates and repeated people may remain because identity metadata are unavailable.

In [ ]:
# Download only when the local data folder is absent.
if not DATA.exists():
    import urllib.request
    archive = ROOT / 'data/yoga107.zip'
    archive.parent.mkdir(parents=True, exist_ok=True)
    if not archive.exists():
        urllib.request.urlretrieve(
            'https://www.kaggle.com/api/v1/datasets/download/shrutisaxena/yoga-pose-image-classification-dataset',
            archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(ROOT / 'data/yoga107_raw')

manifest = pd.read_csv(ART / 'results/manifest.csv')
display(manifest.groupby('split').size().rename('images').to_frame())
print('Classes:', manifest.label.nunique(), '| Clean images:', len(manifest))


In [ ]:
# Input pipeline. Aspect-preserving padding avoids cropping hands and feet.
IMG_SIZE, BATCH_SIZE = 160, 32
augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(.07),
    keras.layers.RandomZoom(.10),
    keras.layers.RandomTranslation(.06, .06),
])

def decode_image(path, label, training=False):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.cast(tf.image.resize_with_pad(image, IMG_SIZE, IMG_SIZE), tf.float32)
    if training:
        image = augmentation(image, training=True)
    return image, label

def make_dataset(split, training=False):
    frame = manifest[manifest.split == split]
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.class_id.values.astype('int32')))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(lambda p,y: decode_image(p,y,training), num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset('train', True)
val_ds = make_dataset('validation')
test_ds = make_dataset('test')


## 2 Transfer learning

I tested ResNet50 because it directly matches the reviewed paper, then EfficientNetB0 as a lighter alternative. Each model first trains a new 107-class head while the ImageNet backbone is frozen. In the second stage, only the top block is unfrozen with a much smaller learning rate. Batch-normalization layers stay frozen to protect their pretrained moving statistics.

In [ ]:
def build_transfer_model(backbone_name='efficientnetb0'):
    if backbone_name == 'resnet50':
        base = keras.applications.ResNet50(weights='imagenet', include_top=False,
            input_shape=(128,128,3), pooling='avg')
        size = 128
    else:
        base = keras.applications.EfficientNetB0(weights='imagenet', include_top=False,
            input_shape=(160,160,3), pooling='avg')
        size = 160
    base.trainable = False
    inputs = keras.Input((size,size,3), name='image')
    features = base(inputs, training=False)
    features = keras.layers.Dropout(.35)(features)
    outputs = keras.layers.Dense(107, activation='softmax')(features)
    return keras.Model(inputs, outputs), base

def two_stage_fit(model, base, train_ds, val_ds, checkpoint):
    callbacks = [
        keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_accuracy', mode='max', save_best_only=True),
        keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=3, restore_best_weights=True),
    ]
    model.compile(keras.optimizers.Adam(8e-4), 'sparse_categorical_crossentropy', metrics=['accuracy'])
    head_history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

    # Fine-tune only the top 35 layers; keep BatchNorm frozen.
    base.trainable = True
    for layer in base.layers[:-35]: layer.trainable = False
    for layer in base.layers[-35:]:
        if isinstance(layer, keras.layers.BatchNormalization): layer.trainable = False
    model.compile(keras.optimizers.Adam(2e-5), 'sparse_categorical_crossentropy', metrics=['accuracy'])
    fine_history = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=callbacks)
    return head_history, fine_history

if RUN_LONG_EXPERIMENTS:
    model, backbone = build_transfer_model('efficientnetb0')
    h1, h2 = two_stage_fit(model, backbone, train_ds, val_ds, ART/'models/efficientnet_best.keras')


## 3 Held-out evaluation and inference time

The Assignment 1 baseline numbers come from its validation-selected `advanced_vanilla` model. Single-image latency uses a warmed-up CPU loop, but the Assignment 1 and Assignment 2 timings were recorded in their respective runs and should be treated as approximate rather than a hardware-neutral benchmark.

In [ ]:
comparison = pd.read_csv(ART/'results/model_comparison.csv')
display(comparison.style.format({'Accuracy':'{:.4f}','Macro F1':'{:.4f}','Median CPU ms':'{:.3f}'}))
display(DisplayImage(filename=str(ART/'figures/model_comparison.png')))
display(DisplayImage(filename=str(ART/'figures/efficientnet_training.png')))


**Observation.** EfficientNetB0 reaches 53.66% test accuracy and 0.5056 macro F1, improving on the custom CNN by 4.65 percentage points and 0.0450 macro F1. ResNet50 reaches only 44.10%, so pretrained weights alone do not guarantee an improvement. EfficientNetB0 is also much larger and slower than the 64×64 custom CNN on CPU; transfer learning benefits predictive performance here, but not latency.

In [ ]:
# Full 107-class test confusion matrix for the best image classifier.
display(DisplayImage(filename=str(ART/'figures/confusion_matrix.png')))
report = pd.read_csv(ART/'results/efficientnet_classification_report.csv', index_col=0)
display(report.loc[manifest.label.unique()].sort_values(['f1-score','recall']).head(12))


## 4 Quantization and Streamlit application

Both the frozen-head and fine-tuned EfficientNetB0 versions are converted to TensorFlow Lite with float16 weight quantization. This reduces storage while retaining float32 inputs and outputs. The Streamlit app provides a model dropdown, image upload, predicted class, confidence, Top-5 table, and confidence chart.

In [ ]:
def export_float16(model, path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    path.write_bytes(converter.convert())

if RUN_LONG_EXPERIMENTS:
    export_float16(keras.models.load_model(ART/'models/efficientnet_head.keras'), ART/'models/efficientnet_head_float16.tflite')
    export_float16(keras.models.load_model(ART/'models/efficientnet_best.keras'), ART/'models/efficientnet_fine_tuned_float16.tflite')

for p in sorted((ART/'models').glob('efficientnet*_float16.tflite')):
    print(p.name, f'{p.stat().st_size/1024/1024:.2f} MiB')


## 5 Feature clustering

I remove the classifier and use the EfficientNetB0 global-average-pooled vectors. After standardization, MiniBatch K-means is tested at several values of k. Silhouette score measures geometric separation; adjusted Rand index compares clusters with pose labels without using those labels during clustering.

In [ ]:
clustering = pd.read_csv(ART/'results/clustering.csv')
display(clustering.style.format({'silhouette_score':'{:.4f}','adjusted_rand_index_vs_pose_labels':'{:.4f}'}))
display(DisplayImage(filename=str(ART/'figures/clustering.png')))


All silhouette scores are negative, and adjusted Rand indices remain near zero. Therefore, the visual feature space does not naturally separate into 107 compact groups matching the yoga labels. Background, viewpoint, person identity, and visually similar poses can dominate Euclidean distance. The tested k values produce 25–125 clusters, but none supports a claim of 107 meaningful natural pose clusters.

## 6 Pose estimation experiment

I use the pretrained MoveNet SinglePose Lightning model to extract 17 keypoints. Coordinates are centered at the hip midpoint and divided by shoulder width. A balanced Random Forest maps the normalized landmark vector to the 107 pose labels. MoveNet is not fine-tuned on Yoga-107; only the small classifier uses the training labels.

In [ ]:
pose = json.loads((ART/'results/pose_estimation.json').read_text())
display(pd.Series(pose, name='value').to_frame())


MoveNet detects a usable pose in 98.43% of the cleaned images. On the 1,149 detected test images, the landmark classifier reaches 66.41% accuracy and 0.6355 macro F1. This result is conditional on successful detection, so it is not directly identical to full-test image-classifier accuracy. Still, it shows that body geometry is more discriminative than generic ImageNet appearance features for many Yoga-107 classes.

## 7 Final approach

My final design would combine normalized MoveNet keypoints with EfficientNetB0 visual features. The keypoint branch supplies explicit body geometry; the image branch handles appearance cues and provides a fallback when pose detection fails or a limb is occluded. I would concatenate both embeddings and train a calibrated classifier, while preserving the same train, validation, and test split. For deployment, I would quantize both branches and report full-test accuracy, macro F1, detection coverage, and end-to-end latency. The current evidence favors the pose branch, but a person or source grouped split is needed before claiming that the system generalizes to new people and environments.

## Submission checks

- The test set is not used for model selection or early stopping.
- Pretrained ImageNet weights are used; no test images enter training.
- Assignment 1 split logic and baseline results are retained.
- Float16 TFLite files are deployed in the Streamlit app.
- Results reported above come from the saved run, not estimated values.
